# QuantAI Predictive Analysis Report

This notebook demonstrates the exploratory data analysis (EDA), feature engineering, model training, and evaluation for the QuantAI stock prediction system.\n
**Course**: Predictive Analysis\n
**Objective**: Predict directional movement and forecast future prices using a fusion of technical indicators and sentiment data.

## 1. Setup & Data Loading

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Prettify plots
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Add root project structure to path to use backend modules
sys.path.append(os.path.abspath('../'))

from backend.data.market_data import fetch_market_data
from backend.features.technical_indicators import compute_all_technical_indicators
from backend.features.regime_features import compute_regime_features

print("Libraries loaded successfully.")

In [ ]:
TICKER = 'AAPL'
df = fetch_market_data(TICKER, years=5)
print(f"Loaded {len(df)} rows for {TICKER}")
df.tail()

## 2. Feature Engineering

In [ ]:
df_features = compute_all_technical_indicators(df)
df_features = compute_regime_features(df_features)
df_features.dropna(inplace=True)

print("Computed Features:", [col for col in df_features.columns if col not in df.columns])
df_features[['Close', 'SMA_20', 'SMA_50', 'RSI_14']].tail()

## 3. Exploratory Data Analysis (EDA)
Analyzing the correlation between engineered features and future returns.

In [ ]:
correlation_matrix = df_features[['Returns', 'Volatility_20d', 'RSI_14', 'MACD', 'MACD_Hist']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title(f"{TICKER} - Feature Correlation Map")
plt.show()

## 4. Backtesting Signals
We use a simplified representation of model prediction via technical regime signals to mock Walk-Forward validation.

In [ ]:
from backend.backtesting.backtest_engine import BacktestEngine, BacktestConfig

signals = pd.Series(0, index=df_features.index)
bullish = (df_features["RSI_14"] < 70) & (df_features["MACD_Hist"] > 0)
bearish = (df_features["RSI_14"] > 30) & (df_features["MACD_Hist"] < 0)
signals[bullish] = 1
signals[bearish] = 0

config = BacktestConfig(initial_capital=100000)
engine = BacktestEngine(config)
result = engine.run(signals, df_features["Close"])

print(result.metrics)

plt.figure(figsize=(12, 5))
plt.plot(result.benchmark_curve, label='Buy & Hold Benchmark', alpha=0.7)
plt.plot(result.equity_curve, label='Strategy Equity Curve', linewidth=2)
plt.title("Strategy vs Benchmark")
plt.legend()
plt.show()